In [1]:
import os
import torch
import torch.nn as nn
from torchvision.transforms import v2 as transforms
import librosa
import numpy as np
from sklearn.metrics import roc_auc_score, mean_squared_error

In [2]:
generator = torch.Generator().manual_seed(42)
np.random.seed(42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
transform = transforms.RandomCrop(size=(128, 256))


class AudioDataset(torch.utils.data.Dataset):
    def __init__(self, audio_dir, train):
        self.audio_dir = audio_dir
        file_list = os.listdir(audio_dir)

        labels = np.zeros(len(file_list), dtype=int) if train else [1 if el[0] == 'a' else 0 for el in file_list]
        self.labels = torch.tensor(labels, dtype=torch.int8).to(device)

        loads = [librosa.load(os.path.join(audio_dir, el), sr=None) for el in file_list]
        spectrograms = [librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128) for audio, sr in loads]
        spect_dbs = [
            torch.tensor(
                librosa.power_to_db(spec, ref=np.max),
                dtype=torch.float32
            )
            for spec in spectrograms
        ]

        spect_dbs = torch.stack(spect_dbs)

        mean = spect_dbs.mean(dim=0)
        std = spect_dbs.std(dim=0)

        self.spect_dbs = (spect_dbs - mean) / std

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return transform(self.spect_dbs[idx]), self.labels[idx]

In [5]:
train_dataset = AudioDataset('archive/dev_data/dev_data/slider/train', train=True)
test_dataset = AudioDataset('archive/dev_data/dev_data/slider/test', train=False)

In [6]:
batch_size = 16

train_set, validation_set = torch.utils.data.random_split(train_dataset, [0.8, 0.2], generator=generator)

train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True
)
validation_loader = torch.utils.data.DataLoader(
    validation_set,
    batch_size=batch_size,
    shuffle=False
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [7]:
class CNNAE(nn.Module):
    def __init__(self):
        super(CNNAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=1, padding=2),   # (32, 128, 256)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                                     # (32, 64, 128)
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),  # (64, 64, 128)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                                     # (64, 32, 64)
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),  # (128, 32, 64)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                                     # (128, 16, 32)
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),   # (64, 32, 64)
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),    # (32, 64, 128)
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=2, stride=2),     # (1, 128, 256)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.encoder(x)
        x = self.decoder(x)
        return x.squeeze(1)


class C1DNNAE(nn.Module):
    def __init__(self):
        super(C1DNNAE, self).__init__()
        # Encoder: input shape (batch, 128, 256)
        self.encoder = nn.Sequential(
            nn.Conv1d(128, 64, kernel_size=5, stride=2, padding=2),   # (batch, 64, 128)
            nn.ReLU(),
            nn.Conv1d(64, 32, kernel_size=5, stride=2, padding=2),    # (batch, 32, 64)
            nn.ReLU(),
            nn.Conv1d(32, 16, kernel_size=5, stride=2, padding=2),    # (batch, 16, 32)
            nn.ReLU(),
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(16, 32, kernel_size=4, stride=2, padding=1),  # (batch, 32, 64)
            nn.ReLU(),
            nn.ConvTranspose1d(32, 64, kernel_size=4, stride=2, padding=1),  # (batch, 64, 128)
            nn.ReLU(),
            nn.ConvTranspose1d(64, 128, kernel_size=4, stride=2, padding=1),  # (batch, 128, 256)
        )

    def forward(self, x):
        # x: (batch, 128, 256) expected
        x = self.encoder(x)
        x = self.decoder(x)
        return x


class C1DNNAE_INV(nn.Module):
    def __init__(self):
        super(C1DNNAE_INV, self).__init__()
        # Encoder: input shape (batch, 256, 128)
        self.encoder = nn.Sequential(
            nn.Conv1d(256, 128, kernel_size=5, stride=2, padding=2),   # (batch, 128, 64)
            nn.ReLU(),
            nn.Conv1d(128, 64, kernel_size=5, stride=2, padding=2),    # (batch, 64, 32)
            nn.ReLU(),
            nn.Conv1d(64, 32, kernel_size=5, stride=2, padding=2),     # (batch, 32, 16)
            nn.ReLU(),
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(32, 64, kernel_size=4, stride=2, padding=1),   # (batch, 64, 32)
            nn.ReLU(),
            nn.ConvTranspose1d(64, 128, kernel_size=4, stride=2, padding=1),  # (batch, 128, 64)
            nn.ReLU(),
            nn.ConvTranspose1d(128, 256, kernel_size=4, stride=2, padding=1),  # (batch, 256, 128)
        )

    def forward(self, x):
        # x: (batch, 128, 256) expected
        x = x.permute(0, 2, 1)
        x = self.encoder(x)
        x = self.decoder(x)
        return x.permute(0, 2, 1)


class LAE(nn.Module):
    def __init__(self):
        super(LAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(128 * 256, 2048),
            nn.ReLU(),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )

        self.decoder = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 2048),
            nn.ReLU(),
            nn.Linear(2048, 128 * 256),
            nn.Tanh()
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.encoder(x)
        x = self.decoder(x)
        return x.view(x.size(0), 128, 256)

In [8]:
def train_one_epoch(model, data_loader, optimizer, criterion):
    model.train()

    total_loss = 0

    for inputs, _ in data_loader:
        inputs = inputs.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        # print(inputs.shape)
        # print(outputs.shape)

        loss = criterion(outputs, inputs)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(data_loader)


def validate(model, data_loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for inputs, _ in data_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, inputs)
            total_loss += loss.item()

    return total_loss / len(data_loader)


def compute_reconstruction_errors(model, data_loader):
    model.eval()
    errors = []
    with torch.no_grad():
        for batch, _ in data_loader:
            batch = batch.to(device)
            reconstructed = model(batch)
            # print(reconstructed.shape, batch.shape)
            error = torch.mean(((reconstructed - batch) ** 2).reshape(batch.size(0), -1), dim=1)
            # print(error.shape)
            errors.extend(error.cpu().numpy())
    return errors


def test(model, data_loader):
    model.eval()

    errors = compute_reconstruction_errors(model, data_loader)

    roc_auc_scores = roc_auc_score(test_dataset.labels.cpu(), errors)
    print(f"ROC AUC Score test: {roc_auc_scores}")

    return roc_auc_scores

In [18]:
def train(lr, weight_decay, epochs):
    model = C1DNNAE_INV().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss = validate(model, validation_loader, criterion)
        test(model, test_loader)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

    return model

In [11]:
train_dataset.spect_dbs.shape, test_dataset.spect_dbs.shape

(torch.Size([2370, 128, 313]), torch.Size([1101, 128, 313]))

In [19]:
model = train(lr=1e-3, weight_decay=1e-8, epochs=50)

ROC AUC Score test: 0.6595734498543486
Epoch 1/50, Train Loss: 0.4557, Validation Loss: 0.3580
ROC AUC Score test: 0.6866500208073242
Epoch 2/50, Train Loss: 0.3120, Validation Loss: 0.3199
ROC AUC Score test: 0.6874365376612568
Epoch 3/50, Train Loss: 0.2924, Validation Loss: 0.3100
ROC AUC Score test: 0.6888347898460259
Epoch 4/50, Train Loss: 0.2881, Validation Loss: 0.3042
ROC AUC Score test: 0.6861131918435289
Epoch 5/50, Train Loss: 0.2881, Validation Loss: 0.3024
ROC AUC Score test: 0.6854057428214732
Epoch 6/50, Train Loss: 0.2866, Validation Loss: 0.3002
ROC AUC Score test: 0.687873491468997
Epoch 7/50, Train Loss: 0.2792, Validation Loss: 0.2987
ROC AUC Score test: 0.6667124427798586
Epoch 8/50, Train Loss: 0.2831, Validation Loss: 0.3121
ROC AUC Score test: 0.6855472326258844
Epoch 9/50, Train Loss: 0.2786, Validation Loss: 0.2974
ROC AUC Score test: 0.685022888056596
Epoch 10/50, Train Loss: 0.2805, Validation Loss: 0.3011
ROC AUC Score test: 0.6886475239284228
Epoch 11/50,

In [ ]:
# path = 'archive/dev_data/dev_data/slider/train'

# loads = [librosa.load(os.path.join(path, el), sr=None) for el in os.listdir(path)]
# print("Loaded")
# spectrograms = [librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128) for audio, sr in loads]
# print("Spectrograms")
# not_flat = [
#     torch.tensor(
#         librosa.power_to_db(spec, ref=np.max),
#         dtype=torch.float32
#     )
#     for spec in spectrograms
# ]
# spect_dbs_not_flat = torch.stack(not_flat).to(device)

# flat = [
#     torch.tensor(
#         librosa.power_to_db(spec, ref=np.max),
#         dtype=torch.float32
#     ).flatten()
#     for spec in spectrograms
# ]
# spect_dbs_flat = torch.stack(flat).to(device)

In [ ]:
# mean_flat = torch.mean(spect_dbs_flat, dim=0)
# std_flat = torch.std(spect_dbs_flat, dim=0)

# end_flat = (spect_dbs_flat - mean_flat) / std_flat

In [ ]:
# mean_not_flat = torch.mean(spect_dbs_not_flat, dim=0)
# std_not_flat = torch.std(spect_dbs_not_flat, dim=0)
# end_not_flat = (spect_dbs_not_flat - mean_not_flat) / std_not_flat
# end2 = end_not_flat.flatten(start_dim=1)

In [ ]:
# torch.allclose(end_flat, end2)

In [ ]:
# spect_dbs_not_flat.shape